# Aqua-Epoch ? Phase 4 Round 1 Analysis

This notebook loads the supplied water-quality dataset, performs a basic quality and exploratory analysis, evaluates a Random Forest classifier for the dataset's `Potability` label, and runs pH/TDS regression baselines.

**Important:** For this analysis, label 1 is interpreted as safe and 0 as unsafe. Confirm the label definition with the data provider. A model prediction is not a BIS compliance determination or a substitute for laboratory testing.


In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, mean_absolute_error, mean_squared_error,
                             precision_score, r2_score, recall_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")
warnings.filterwarnings("ignore", category=FutureWarning)
print("Imports complete.")


In [ ]:
# Load the CSV located beside this notebook.
csv_files = sorted(Path.cwd().glob("*.csv"))
if not csv_files:
    raise FileNotFoundError("No CSV found. Place the Aqua-Epoch dataset beside this notebook.")
dataset_path = csv_files[0]
df_raw = pd.read_csv(dataset_path)
df = df_raw.dropna(how="all").copy()
if "Potability" not in df.columns:
    raise ValueError("Dataset is missing the Potability target column.")

labeled_df = df.dropna(subset=["Potability"]).copy()
labeled_df["Potability"] = pd.to_numeric(labeled_df["Potability"], errors="coerce")
labeled_df = labeled_df.dropna(subset=["Potability"])
labeled_df["Potability"] = labeled_df["Potability"].astype(int)
feature_columns = [c for c in labeled_df.select_dtypes(include=np.number).columns if c != "Potability"]

print(f"Dataset: {dataset_path.name}")
print(f"Original shape: {df_raw.shape}; non-empty rows: {len(df):,}; labeled rows: {len(labeled_df):,}")
print("Missing values in non-empty records:")
display(df.isna().sum().sort_values(ascending=False).rename("missing_count").to_frame())
print("Potability label counts (interpretation must be confirmed):")
display(labeled_df["Potability"].value_counts().sort_index().rename_axis("label").to_frame("count"))
display(labeled_df.head())


In [ ]:
# Exploratory views: class balance, missingness, and correlation.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(data=labeled_df, x="Potability", ax=axes[0])
axes[0].set_title("Potability Label Counts")
axes[0].set_xlabel("Dataset label (confirm meaning)")
axes[0].set_ylabel("Samples")
missing = (df[feature_columns].isna().mean() * 100).sort_values(ascending=False)
missing = missing[missing > 0]
if len(missing):
    sns.barplot(x=missing.values, y=missing.index, ax=axes[1], color="steelblue")
    axes[1].set_title("Feature Missingness (%)")
else:
    axes[1].text(0.5, 0.5, "No missing features", ha="center")
    axes[1].set_axis_off()
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 9))
sns.heatmap(labeled_df[feature_columns + ["Potability"]].corr(numeric_only=True), cmap="coolwarm", center=0)
plt.title("Correlation Matrix (descriptive, not causal)")
plt.tight_layout()
plt.show()


In [ ]:
# Classification baseline; imputation is fitted on training data only.
X = labeled_df[feature_columns].replace([np.inf, -np.inf], np.nan).copy()
if "pH" in X:
    X.loc[(X["pH"] < 0) | (X["pH"] > 14), "pH"] = np.nan
for col in ("Hardness", "Fluoride"):
    if col in X:
        X.loc[X[col] < 0, col] = np.nan
y = labeled_df["Potability"]
if y.nunique() != 2:
    raise ValueError(f"Expected binary target; found {sorted(y.unique())}")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
classifier = make_pipeline(
    SimpleImputer(strategy="median"),
    RandomForestClassifier(n_estimators=400, random_state=RANDOM_STATE,
                           class_weight="balanced", n_jobs=-1),
)
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)
metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision (class 1)", "Recall (class 1)", "F1 (class 1)", "Macro F1"],
    "Score": [accuracy_score(y_test, y_pred),
              precision_score(y_test, y_pred, pos_label=1, zero_division=0),
              recall_score(y_test, y_pred, pos_label=1, zero_division=0),
              f1_score(y_test, y_pred, pos_label=1, zero_division=0),
              f1_score(y_test, y_pred, average="macro", zero_division=0)]
})
print(f"Train rows: {len(X_train):,}; test rows: {len(X_test):,}")
display(metrics.style.format({"Score": "{:.4f}"}))
print(classification_report(y_test, y_pred, zero_division=0))


In [ ]:
# Confusion matrix and model-specific feature importance.
labels = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
plt.xlabel("Predicted label")
plt.ylabel("Actual label")
plt.title("Classification Confusion Matrix")
plt.tight_layout()
plt.show()

rf = classifier.named_steps["randomforestclassifier"]
importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
display(importance.rename("importance").to_frame())
plt.figure(figsize=(8, 6))
sns.barplot(x=importance.head(15).values, y=importance.head(15).index, color="teal")
plt.title("Random Forest Importance (not causal)")
plt.xlabel("Impurity-based importance")
plt.tight_layout()
plt.show()


In [ ]:
# Regression baselines for pH and TDS. Each target is excluded from its own inputs.
regression_rows = []
regression_results = {}
for target in ("pH", "TDS"):
    target_values = pd.to_numeric(df[target], errors="coerce")
    valid = target_values.notna() & np.isfinite(target_values)
    if target == "pH":
        valid &= target_values.between(0, 14)
    target_data = df.loc[valid]
    y_reg = pd.to_numeric(target_data[target], errors="coerce")
    reg_cols = [c for c in feature_columns if c != target]
    X_reg = target_data[reg_cols].replace([np.inf, -np.inf], np.nan).copy()
    if "pH" in X_reg:
        X_reg.loc[(X_reg["pH"] < 0) | (X_reg["pH"] > 14), "pH"] = np.nan
    for col in ("Hardness", "Fluoride"):
        if col in X_reg:
            X_reg.loc[X_reg[col] < 0, col] = np.nan
    Xr_train, Xr_test, yr_train, yr_test = train_test_split(
        X_reg, y_reg, test_size=0.20, random_state=RANDOM_STATE
    )
    regressor = make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
    )
    regressor.fit(Xr_train, yr_train)
    prediction = regressor.predict(Xr_test)
    regression_results[target] = (yr_test, prediction)
    regression_rows.append({
        "Target": target, "Test rows": len(Xr_test),
        "MAE": mean_absolute_error(yr_test, prediction),
        "RMSE": mean_squared_error(yr_test, prediction) ** 0.5,
        "R2": r2_score(yr_test, prediction),
    })
regression_metrics = pd.DataFrame(regression_rows)
display(regression_metrics.style.format({"MAE": "{:.3f}", "RMSE": "{:.3f}", "R2": "{:.3f}"}))
for target, (actual, predicted) in regression_results.items():
    plt.figure(figsize=(5, 5))
    plt.scatter(actual, predicted, alpha=0.25, s=12)
    low = min(float(actual.min()), float(np.min(predicted)))
    high = max(float(actual.max()), float(np.max(predicted)))
    plt.plot([low, high], [low, high], "r--")
    plt.xlabel(f"Actual {target}")
    plt.ylabel(f"Predicted {target}")
    plt.title(f"{target}: Actual vs Predicted")
    plt.tight_layout()
    plt.show()


## Limitations and next steps

- Classification scores evaluate agreement with this dataset's label, not confirmed BIS compliance.
- This is a single random holdout. Validate with cross-validation and an independent time/location-based dataset before operational use.
- The dataset contains missing and suspicious readings. Confirm units, label provenance, laboratory methods, and sensor tolerances with the data provider.
- Outlier treatment here is intentionally limited; only pH outside 0?14 and negative hardness/fluoride are marked missing.
- Random Forest feature importance is model-dependent and not causal. Regression estimates are not substitutes for lab measurements.
